# 03 — EDA Multivariate: Group Differences


In [ ]:
import sys
from pathlib import Path
REPO = Path.cwd()
for candidate in [REPO, REPO.parent, REPO.parent.parent]:
    if (candidate / "src" / "neuro").exists():
        REPO = candidate
        break
sys.path.insert(0, str(REPO / "src"))
import os
os.chdir(REPO)
%matplotlib inline
import matplotlib.pyplot as plt
import seaborn as sns
sns.set_theme(style="whitegrid", context="notebook")


In [ ]:

import numpy as np
import pandas as pd
from sklearn.decomposition import PCA
from neuro.bids import validate_bids
from neuro.features import get_schaefer_masker, extract_roi_timeseries, connectivity_matrix
from neuro.mlflow_utils import start_run
from neuro import viz

with start_run("03_eda_multivariate"):
    report = validate_bids()
    runs = report["runs"][report["runs"]["bold_exists"]].head(4)
    masker = get_schaefer_masker(n_rois=100)
    mlflow.log_param("n_rois", 100)
    conn_mats = []
    for _, row in runs.iterrows():
        ts = extract_roi_timeseries(row["bold_path"], masker)
        conn_mats.append(connectivity_matrix(ts))
    mean_conn = np.mean(conn_mats, axis=0)
    viz.plot_connectivity_heatmap(mean_conn, "Mean ROI connectivity (subset)")
    plt.show()


In [ ]:

from collections import defaultdict
sub_conn = defaultdict(list)
for _, row in report["runs"][report["runs"]["bold_exists"]].iterrows():
    ts = extract_roi_timeseries(row["bold_path"], masker)
    sub_conn[(row["subject"], row["group_short"])].append(connectivity_matrix(ts))

records = []
for (sub, grp), mats in sub_conn.items():
    vec = np.mean(mats, axis=0)[np.triu_indices_from(np.mean(mats, axis=0), k=1)]
    records.append({"subject": sub, "group": grp, "features": vec})
feat_df = pd.DataFrame(records)
X = np.stack(feat_df["features"].values)
pca = PCA(n_components=2, random_state=42)
coords = pca.fit_transform(X)
plot_df = feat_df[["subject", "group"]].copy()
plot_df["PC1"], plot_df["PC2"] = coords[:, 0], coords[:, 1]
mlflow.log_metric("pca_explained_var_pc1", float(pca.explained_variance_ratio_[0]))
fig, ax = plt.subplots(figsize=(7, 5))
sns.scatterplot(data=plot_df, x="PC1", y="PC2", hue="group", style="group", s=120, ax=ax)
ax.set_title("PCA of ROI connectivity (per subject)")
plt.show()

mdd = [np.mean(v, axis=0) for (_, g), v in sub_conn.items() if g == "MDD"][:5]
nd = [np.mean(v, axis=0) for (_, g), v in sub_conn.items() if g == "ND"][:5]
if mdd and nd:
    diff = np.mean(mdd, axis=0) - np.mean(nd, axis=0)
    viz.plot_connectivity_heatmap(diff, "MDD − ND connectivity diff")
    plt.show()
